# CPDE-7 Extraction Testing with CPDE7LLMService

This notebook tests the `CPDE7LLMService` class which encapsulates all 7 dimensions of the Conversational Profiling Data Extraction (CPDE-7) framework.

## Dimensions
1. Core Identity
2. Opinions & Views
3. Preferences & Patterns
4. Desires, Wishes, Hopes & Needs
5. Life Narrative
6. Events & Involvement
7. Entities & Relationships

In [ ]:
# Load environment variables
from dotenv import load_dotenv
import os

# Load from project root .env file
load_dotenv("../../../../../.env", override=True)

# Verify API key is loaded
print(f"OPENAI_API_KEY loaded: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")

In [ ]:
# Import the CPDE7LLMService
from chatforge.services.profiling_data_extraction import (
    CPDE7LLMService,
    CPF7_DIMENSIONS,
)
import json

print(f"Available dimensions: {CPF7_DIMENSIONS}")

In [ ]:
# Initialize the service
cpde_service = CPDE7LLMService(
    provider="openai",
    model_name="gpt-4o-mini",
    temperature=0,
)

print(f"Service initialized: {cpde_service.model_info}")

In [ ]:
# Helper function to display results
def display_results(result, dimension_name: str):
    """Pretty print extraction results."""
    # Get the dimension result (first attribute)
    dim_result = getattr(result, list(result.model_fields.keys())[0])
    
    print(f"\n{'='*60}")
    print(f"{dimension_name.upper()}")
    print(f"{'='*60}")
    print(f"Has content: {dim_result.has_content}")
    print(f"Items extracted: {len(dim_result.items)}")
    
    for i, item in enumerate(dim_result.items, 1):
        print(f"\n--- Item {i} ---")
        print(f"  Source: {item.source_message_id}")
        print(f"  Quote: \"{item.source_quote}\"")
        # Print other fields
        for field_name, field_value in item.model_dump().items():
            if field_name not in ['source_message_id', 'source_quote'] and field_value is not None:
                print(f"  {field_name}: {field_value}")

## Test Messages

A rich set of messages containing data for all 7 dimensions.

In [ ]:
# Comprehensive test messages covering all 7 dimensions
test_messages = """
Message ID: msg_001
Content: Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic.

Message ID: msg_002
Content: I think remote work is the future, but companies need better async communication tools.

Message ID: msg_003
Content: I always code better at night and I can't work without my noise-canceling headphones.

Message ID: msg_004
Content: I really want to transition into AI/ML and I hope to lead a team someday.

Message ID: msg_005
Content: I lived in Tokyo for 3 years after college and it completely changed my worldview.

Message ID: msg_006
Content: I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy.

Message ID: msg_007
Content: My mentor Sarah from my previous job at Microsoft still helps me weekly. My dog Max keeps me company while I work.
"""

print("Test messages loaded:")
print(test_messages)

---
## 1. Core Identity Extraction

In [ ]:
print("Extracting Core Identity...")
result_identity = await cpde_service.extract_core_identity(test_messages)
display_results(result_identity, "Core Identity")

---
## 2. Opinions & Views Extraction

In [ ]:
print("Extracting Opinions & Views...")
result_opinions = await cpde_service.extract_opinions_views(test_messages)
display_results(result_opinions, "Opinions & Views")

---
## 3. Preferences & Patterns Extraction

In [ ]:
print("Extracting Preferences & Patterns...")
result_preferences = await cpde_service.extract_preferences_patterns(test_messages)
display_results(result_preferences, "Preferences & Patterns")

---
## 4. Desires, Wishes, Hopes & Needs Extraction

In [ ]:
print("Extracting Desires & Needs...")
result_desires = await cpde_service.extract_desires_needs(test_messages)
display_results(result_desires, "Desires & Needs")

---
## 5. Life Narrative Extraction

In [ ]:
print("Extracting Life Narrative...")
result_narrative = await cpde_service.extract_life_narrative(test_messages)
display_results(result_narrative, "Life Narrative")

---
## 6. Events & Involvement Extraction

In [ ]:
print("Extracting Events...")
result_events = await cpde_service.extract_events(test_messages)
display_results(result_events, "Events & Involvement")

---
## 7. Entities & Relationships Extraction

In [ ]:
print("Extracting Entities & Relationships...")
result_entities = await cpde_service.extract_entities_relationships(test_messages)
display_results(result_entities, "Entities & Relationships")

---
## Summary: All 7 Dimensions

In [ ]:
# Collect all results
all_results = {
    'core_identity': result_identity,
    'opinions_views': result_opinions,
    'preferences_patterns': result_preferences,
    'desires_needs': result_desires,
    'life_narrative': result_narrative,
    'events': result_events,
    'entities_relationships': result_entities,
}

# Summary view
print("\n" + "="*60)
print("EXTRACTION SUMMARY")
print("="*60)

total_items = 0
for dim_name, result in all_results.items():
    dim_result = getattr(result, list(result.model_fields.keys())[0])
    count = len(dim_result.items)
    total_items += count
    status = "" if dim_result.has_content else "(empty)"
    print(f"  {dim_name:25} : {count:3} items {status}")

print(f"\n  {'TOTAL':25} : {total_items:3} items")

---
## Using extract_all() Method

The service also provides a convenient `extract_all()` method to extract all dimensions at once.

In [ ]:
# Extract all dimensions at once (sequential)
print("Extracting ALL dimensions using extract_all()...")
full_result = await cpde_service.extract_all(test_messages)

print("\n" + "="*60)
print("EXTRACT_ALL SUMMARY")
print("="*60)

total = 0
for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(full_result, dim_name, None)
    if dim_result:
        count = len(dim_result.items)
        total += count
        print(f"  {dim_name:25} : {count:3} items")
    else:
        print(f"  {dim_name:25} : None")

print(f"\n  {'TOTAL':25} : {total:3} items")

---
## Using extract_all() with Parallel Execution

You can also extract dimensions in parallel for better performance.

In [ ]:
# Extract selected dimensions in PARALLEL
import time

print("Extracting selected dimensions (core_identity, events, entities_relationships) in PARALLEL...")
start = time.time()
partial_result = await cpde_service.extract_all(
    test_messages,
    dimensions=["core_identity", "events", "entities_relationships"],
    parallel=True  # Run concurrently!
)
elapsed = time.time() - start

print(f"\nCompleted in {elapsed:.2f}s")
print("\n" + "="*60)
print("PARALLEL EXTRACTION SUMMARY")
print("="*60)

for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(partial_result, dim_name, None)
    if dim_result:
        count = len(dim_result.items)
        print(f"  {dim_name:25} : {count:3} items")
    else:
        print(f"  {dim_name:25} : (not extracted)")

---
## Using extract_all_7() - Single LLM Call

Extract all 7 dimensions in a single LLM call using the combined prompt. More efficient but may be less accurate for complex messages.

In [ ]:
# Extract all 7 dimensions in ONE LLM call
import time

print("Extracting ALL 7 dimensions in a SINGLE LLM call...")
start = time.time()
all_7_result = await cpde_service.extract_all_7(test_messages)
elapsed = time.time() - start

print(f"\nCompleted in {elapsed:.2f}s")
print("\n" + "="*60)
print("EXTRACT_ALL_7 (SINGLE CALL) SUMMARY")
print("="*60)

total = 0
for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(all_7_result, dim_name, None)
    if dim_result:
        count = len(dim_result.items)
        total += count
        status = "✓" if dim_result.has_content else "○"
        print(f"  {status} {dim_name:25} : {count:3} items")
    else:
        print(f"  ✗ {dim_name:25} : None")

print(f"\n    {'TOTAL':25} : {total:3} items")
print(f"\n(Single call is more efficient than 7 separate calls)")

In [ ]:
# Show detailed output from extract_all_7 for entities dimension
print("Entities extracted in single-call mode:")
print("="*60)

if all_7_result.entities_relationships.has_content:
    for i, item in enumerate(all_7_result.entities_relationships.items, 1):
        print(f"\n--- Entity {i} ---")
        print(f"  Source: {item.source_message_id}")
        print(f"  Name: {item.name}")
        print(f"  Type: {item.entity_type}")
        print(f"  Properties: {item.mentioned_properties}")
        print(f"  Relationships: {item.relationship_indicators}")
else:
    print("No entities extracted")

---
## Using extract_dimension() Method

Extract a single dimension by name.

In [ ]:
# Extract by dimension name
print("Extracting 'events' dimension by name...")
events_result = await cpde_service.extract_dimension(test_messages, "events")
display_results(events_result, "Events (via extract_dimension)")

---
## Edge Case Testing

Test messages that should NOT produce extractions for certain dimensions.

In [ ]:
# Edge case messages
edge_case_messages = """
Message ID: edge_001
Content: This coffee is cold and the meeting was pointless.

Message ID: edge_002
Content: I had lunch and then went to the store.

Message ID: edge_003
Content: Apple announced a new iPhone yesterday.
"""

print("Edge case messages (should extract minimal data):")
print(edge_case_messages)

In [ ]:
# Test edge cases using extract_all (parallel for speed)
print("Testing edge cases on all 7 dimensions (parallel)...\n")

edge_result = await cpde_service.extract_all(edge_case_messages, parallel=True)

for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(edge_result, dim_name, None)
    if dim_result:
        count = len(dim_result.items)
        if count == 0:
            print(f"  {dim_name:25} : 0 items (correct!)")
        else:
            print(f"  {dim_name:25} : {count} items (review needed)")
            for item in dim_result.items:
                quote = item.source_quote[:50] + "..." if len(item.source_quote) > 50 else item.source_quote
                print(f"    - {quote}")

---
## Raw JSON Output

In [ ]:
# Show raw JSON for Core Identity
print("Raw JSON output (Core Identity):")
print(json.dumps(result_identity.model_dump(), indent=2))

In [ ]:
# Show raw JSON for Entities
print("Raw JSON output (Entities & Relationships):")
print(json.dumps(result_entities.model_dump(), indent=2))